# Chapter 3:Coding Attention Mechanisms(编码注意力机制)

## 3.3:Attending to different parts of the input with with self-attention

### 3.3.1:A simple self-attention mechanism without trainable weights

In [1]:
import torch

inputs=torch.tensor(
    [[0.43,0.15,0.89], # Your     (x^1)
     [0.55,0.87,0.66], # journey  (x^2)
     [0.57,0.85,0.64], # starts   (x^3)
     [0.22,0.58,0.33], # with     (x^4)
     [0.77,0.25,0.10], # one      (x^5)
     [0.05,0.80,0.55]] # step     (x^6)
)

In [2]:
input_query = inputs[1]
print(input_query)

tensor([0.5500, 0.8700, 0.6600])


In [3]:
input_1=inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [4]:
0.55*0.43+0.87*0.15+ 0.66*0.89

0.9544

In [5]:
torch.dot(input_query,input_1)# 点积

tensor(0.9544)

In [6]:
i=2

res=torch.dot(inputs[i],input_query)
res

tensor(1.4754)

In [7]:
inputs.shape[0]

6

In [8]:
# torch.empty(inputs.shape[0])
torch.zeros(inputs.shape[0])

tensor([0., 0., 0., 0., 0., 0.])

In [9]:
query=inputs[1] # 2nd input token is the query

attn_scores_2=torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
    attn_scores_2[i]=torch.dot(x_i,input_query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [10]:
attn_weights_2_tmp=attn_scores_2/attn_scores_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [11]:
attn_weights_2_tmp.sum()

tensor(1.0000)

In [12]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [13]:
attn_weights_2=torch.softmax(attn_scores_2,dim=0)

In [14]:
torch.zeros(query.shape)

tensor([0., 0., 0.])

In [15]:
query=inputs[1] # 2nd input token is the query

context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    # print(f"{attn_weights_2[i]} ---> {inputs[i]}")
    context_vec_2+=attn_weights_2[i]*inputs[i]

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [16]:
for i,x_i in enumerate(inputs):
    print(i,inputs[i])

0 tensor([0.4300, 0.1500, 0.8900])
1 tensor([0.5500, 0.8700, 0.6600])
2 tensor([0.5700, 0.8500, 0.6400])
3 tensor([0.2200, 0.5800, 0.3300])
4 tensor([0.7700, 0.2500, 0.1000])
5 tensor([0.0500, 0.8000, 0.5500])


In [17]:
inputs=torch.tensor(
    [[0.43,0.15,0.89], # Your     (x^1)
     [0.55,0.87,0.66], # journey  (x^2)
     [0.57,0.85,0.64], # starts   (x^3)
     [0.22,0.58,0.33], # with     (x^4)
     [0.77,0.25,0.10], # one      (x^5)
     [0.05,0.80,0.55]] # step     (x^6)
)

### 3.3.2:Computing attention weights for all input tokens

In [18]:
attn_scores=inputs @ inputs.T
attn_weights=torch.softmax(attn_scores,dim=1)
all_context_vecs=attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## 3.4:Implementing self-attention with trainable weights

### 3.4.1:Computing the attention weights step by step

In [19]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2

In [20]:
torch.manual_seed(123)

W_query=torch.nn.Parameter(torch.rand(d_in,d_out))
W_key=torch.nn.Parameter(torch.rand(d_in,d_out))
W_value=torch.nn.Parameter(torch.rand(d_in,d_out))

In [21]:
query_2=x_2 @ W_query

query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [22]:
keys=inputs @ W_key
values=inputs @ W_value

keys.shape

torch.Size([6, 2])

In [23]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [24]:
keys_2=keys[1]
attn_score_22=torch.dot(query_2,keys_2)

In [25]:
attn_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [26]:
attn_scores_2=query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [27]:
d_k=keys.shape[1]

attn_weights_2=torch.softmax(attn_scores_2 / d_k**0.5 ,dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [28]:
attn_weights_2.sum()

tensor(1., grad_fn=<SumBackward0>)

In [29]:
context_vec_2=attn_weights_2 @ values

context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

### 3.4.2:Implementing a compact SelfAttention class

In [30]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query=torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_key=torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_value=torch.nn.Parameter(torch.rand(d_in,d_out))


    def forward(self,x):
        queries=inputs @ W_query
        keys=inputs @ W_key
        values=inputs @ W_value

        attn_scores=queries @ keys.T
        attn_weights=torch.softmax(attn_scores / d_k**0.5 ,dim=-1)
        context_vec=attn_weights @ values
        
        return context_vec

torch.manual_seed(123)
sa_v1=SelfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [31]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [32]:
m=torch.nn.Linear(2,3)
m.weight

Parameter containing:
tensor([[-0.1668,  0.2270],
        [ 0.5000,  0.1317],
        [ 0.1934,  0.6825]], requires_grad=True)

In [33]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self,d_in,d_out,qkv_bias=False):
        super().__init__()
        self.W_query=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=torch.nn.Linear(d_in,d_out,bias=qkv_bias)


    def forward(self,x):
        queries=self.W_query(inputs)
        keys=self.W_key(inputs)
        values=self.W_value(inputs)

        attn_scores=queries @ keys.T
        attn_weights=torch.softmax(attn_scores / d_k**0.5 ,dim=-1)
        context_vec=attn_weights @ values
        
        return context_vec

torch.manual_seed(789)
sa_v2=SelfAttention_v2(d_in,d_out)
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## 3.5:Hiding future words with causal attention

### 3.5.1:Applying a causal attention mask

In [34]:
# Your journey starts with one step


In [35]:
# Your -> journey

In [36]:
queries=sa_v2.W_query(inputs)
keys=sa_v2.W_key(inputs)
values=sa_v2.W_value(inputs)

attn_scores=queries @ keys.T
attn_weights=torch.softmax(attn_scores / d_k**0.5 ,dim=-1)

In [37]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [38]:
context_length=attn_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [39]:
masked_simple=attn_weights*mask_simple
masked_simple

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [40]:
row_sums=masked_simple.sum(dim=-1,keepdim=True)
masked_simple_norm=masked_simple/row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [41]:
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked=attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [42]:
torch.exp(torch.tensor(-9999999999999))

tensor(0.)

In [43]:
torch.exp(torch.tensor(float("-inf")))

tensor(0.)

In [44]:
# attn_weights=torch.softmax(attn_scores / d_k**0.5 ,dim=-1)
attn_weights=torch.softmax(masked / keys.shape[-1]**0.5,dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2:Masking additional attention weights with dropout

In [45]:
torch.manual_seed(124)

layer=torch.nn.Dropout(0.5)


In [46]:
example=torch.ones(6,6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [47]:
layer(example)

tensor([[2., 2., 2., 2., 2., 0.],
        [2., 2., 0., 0., 2., 2.],
        [0., 0., 0., 2., 0., 0.],
        [0., 2., 2., 2., 2., 2.],
        [2., 2., 2., 0., 0., 2.],
        [0., 0., 0., 2., 0., 0.]])

In [48]:
dropout_rate=0.5
1/(1-dropout_rate)

2.0

In [49]:
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.0000, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.3966, 0.3968, 0.0000, 0.0000, 0.0000],
        [0.3869, 0.3327, 0.3331, 0.0000, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)

### 3.5.3:Implementing a compact causal self-attention class

In [50]:
batch=torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [51]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.W_query=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout=torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b,num_tokens,d_in=x.shape
        # x = bach,2 x 6 x 3
        queries=self.W_query(x)
        keys=self.W_key(x)
        values=self.W_value(x)

        attn_scores=queries @ keys.transpose(1,2)# Changed transpose
        attn_scores.masked_fill_(  # New, _ops are in-place
            self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)  #`num_tokens` to account for cases where the
        attn_weights=torch.softmax(
            attn_scores/keys.shape[-1]**0.5,dim=-1
        )
        attn_weights=self.dropout(attn_weights) # New

        
        context_vec=attn_weights @ values
        return context_vec

torch.manual_seed(789)

context_length=batch.shape[1]
dropout=0.0
ca=CausalAttention(d_in,d_out,context_length,dropout)
ca(batch)



tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

In [52]:
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [55]:
batch.shape[1]

6

In [56]:
batch.shape

torch.Size([2, 6, 3])

## 3.6:Extending single-head attention to multi-head attention

### 3.6.1:Stacking multiple single-head attention layers

In [63]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads=2,qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([
            CausalAttention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)
        ])

    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)

torch.manual_seed(123)

context_length=batch.shape[1]
d_in,d_out=3,2

mha=MultiHeadAttentionWrapper(d_in,d_out,context_length,dropout=0.0,num_heads=2)
mha(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

### 3.6.2:Implementing multi-head attention with weight splits

In [65]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out%num_heads==0), \
            "d_out must be divisible by num_heads"

        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out // num_heads # Reducce the projection dim to match desired output dim

        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj=nn.Linear(d_out,d_out) #Linear layer to combine head outputs
        self.dropout=nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),
                        diagonal=1)
        )

    def forward(self,x):
        b,num_tokens,d_in=x.shape

        keys=self.W_key(x) # Shape: (b,num_tokens,d_out)
        queries=self.W_query(x)
        values=self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b,num_tokens,d_out) -> (b,num_tokens,num_heads,head_dim)
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b,num_tokens,self.num_heads,self.head_dim)
        queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)

        # Transpose: (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim)
        keys=keys.transpose(1,2)
        queries=queries.transpose(1,2)
        values=values.transpose(1,2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores=queries @ keys.transpose(2,3) # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool,-torch.inf)

        attn_weights=torch.softmax(attn_scores / keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)

        # Shape: (b,num_tokens,num_heads,head_dim)
        context_vec=(attn_weights @ values).transpose(1,2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec=context_vec.reshape(b,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size,context_length,d_in=batch.shape
d_out=4
mha=MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)

context_vecs=mha(batch)

print(context_vecs)
print("context_vecs.shape:",context_vecs.shape)

tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
